# 02 — Data quality and conservative cleaning

Input is `data/synthetic_ivf_raw.csv`; output is `data/synthetic_ivf_cleaned.csv`. The pipeline removes exact duplicate rows, standardizes categories, converts numeric/date fields, removes malformed cycle-date rows, sets impossible range/count values to missing, preserves unknown outcomes, and retains statistical IQR outliers.

In [1]:
from src.data_quality import clean_data, iqr_screen
raw, clean, before, after, raw_profile, clean_profile = clean_data()
print("BEFORE:", before)
print("AFTER:", {k:v for k,v in after.items() if k not in ["invalid_range_counts", "logical_count_violations"]})
print("invalid ranges:", after["invalid_range_counts"])
print("logical violations:", after["logical_count_violations"])
print("remaining duplicate rows:", clean.duplicated().sum())

BEFORE: {'raw_rows': 3005, 'raw_columns': 29, 'exact_duplicate_rows': 5, 'missing_cells': 375}
AFTER: {'clean_rows': 2998, 'clean_columns': 30, 'rows_removed_exact_duplicates': 5, 'rows_removed_invalid_identity_or_date': 2, 'invalid_dates_after_dedup': 2, 'missing_cells_after_cleaning': 380, 'unique_patients': 2100, 'unique_cycles': 2998, 'transfer_cycles': 2910, 'known_clinical_outcomes': 2876, 'missing_clinical_outcomes_among_transfers': 34, 'date_min': '2022-01-01', 'date_max': '2025-12-30'}
invalid ranges: {'patient_age': 2, 'infertility_duration_months': 0, 'bmi': 2, 'afc': 0, 'amh_ng_ml': 0, 'stimulation_days': 0, 'estradiol_pg_ml': 0, 'progesterone_ng_ml': 0, 'total_oocytes': 0, 'mature_oocytes': 0, 'fertilization_count': 0, 'embryos_formed': 0, 'day_of_embryo_transfer': 0, 'embryos_transferred': 0, 'blastocyst_indicator': 0, 'clinical_pregnancy_outcome': 0, 'biochemical_pregnancy_outcome': 0, 'live_birth_outcome': 0, 'endometrial_thickness_mm': 0, 'treatment_cycle_cost_inr': 0}

## Missingness by column and IQR flags
IQR flags are not deleted. Missing outcome values include non-transfer cycles and unknown follow-up; use transfer cycles with observed outcome for rates.

In [2]:
from IPython.display import display
display(raw_profile.sort_values("missing_count",ascending=False).head(12))
display(clean_profile.sort_values("missing_count",ascending=False).head(12))
iqr = iqr_screen(clean)
display(iqr.sort_values("flagged_count",ascending=False).head(12).round(3))
print("All flagged values retained in cleaned data; inspect, do not auto-delete.")

,dtype,missing_count,missing_pct,unique_non_null
biochemical_pregnancy_outcome,float64,122,4.060,2
live_birth_outcome,float64,122,4.060,2
clinical_pregnancy_outcome,float64,122,4.060,2
amh_ng_ml,float64,3,0.100,715
progesterone_ng_ml,float64,2,0.067,241
treatment_cycle_cost_inr,float64,2,0.067,2947
endometrial_thickness_mm,float64,2,0.067,100
patient_age,int64,0,0.000,26
infertility_duration_months,int64,0,0.000,111
cycle_date,object,0,0.000,1259


,dtype,missing_count,missing_pct,unique_non_null
live_birth_outcome,float64,122,4.069,2
clinical_pregnancy_outcome,float64,122,4.069,2
biochemical_pregnancy_outcome,float64,122,4.069,2
amh_ng_ml,float64,3,0.100,715
bmi,float64,2,0.067,198
patient_age,float64,2,0.067,24
endometrial_thickness_mm,float64,2,0.067,100
progesterone_ng_ml,float64,2,0.067,241
treatment_cycle_cost_inr,float64,2,0.067,2945
mature_oocytes,float64,1,0.033,22


,column,q1,q3,iqr,lower_fence,upper_fence,flagged_count,flagged_pct_nonmissing
14,embryos_transferred,1.00,1.00,0.00,1.00,1.00,585,19.513
5,amh_ng_ml,2.12,4.30,2.18,-1.15,7.57,131,4.374
8,progesterone_ng_ml,0.57,1.11,0.54,-0.24,1.92,102,3.405
2,infertility_duration_months,16.00,41.00,25.00,-21.50,78.50,102,3.402
1,patient_age,31.00,36.00,5.00,23.50,43.50,92,3.071
9,total_oocytes,8.00,13.00,5.00,0.50,20.50,53,1.768
12,embryos_formed,3.00,6.00,3.00,-1.50,10.50,41,1.368
16,endometrial_thickness_mm,8.00,10.20,2.20,4.70,13.50,32,1.068
10,mature_oocytes,5.00,10.00,5.00,-2.50,17.50,20,0.667
20,treatment_cycle_cost_inr,162310.25,197041.25,34731.00,110213.75,249137.75,17,0.567


All flagged values retained in cleaned data; inspect, do not auto-delete.
